# 04 — Building a Transformer Encoder

    **Companion chapter:** `04-transformer-encoder(2).md`

    ## Learning goals

    - Use token embeddings and sinusoidal positions.
- Calculate residual connections and LayerNorm.
- Implement a position-wise feed-forward network.
- Assemble one Transformer encoder layer.
- Stack layers while tracking tensor shapes.

    ## How to use this notebook

    Run the cells from top to bottom. Read the comments, change small values, and
    rerun the cell. Every notebook ends with practice prompts that can become
    GitHub issues, exercises, or discussion questions.

In [1]:
from __future__ import annotations

import math
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## 1. Token embeddings

An embedding table maps each vocabulary ID to a trainable `d_model` vector.

In [2]:
vocabulary = ["<PAD>", "من", "کتاب", "را", "خواندم"]
stoi = {token: index for index, token in enumerate(vocabulary)}

token_ids = torch.tensor([[stoi[token] for token in ["من", "کتاب", "را", "خواندم"]]])
embedding = nn.Embedding(len(vocabulary), embedding_dim=8, padding_idx=0)

token_vectors = embedding(token_ids)
print("Token IDs:", token_ids)
print("Embedding tensor:", tuple(token_vectors.shape))

Token IDs: tensor([[1, 2, 3, 4]])
Embedding tensor: (1, 4, 8)


## 2. Sinusoidal positional encodings

The same position vector is added to every token occurring at that position.

In [3]:
def sinusoidal_positional_encoding(
    max_length: int,
    d_model: int,
    device: torch.device | str = "cpu",
) -> torch.Tensor:
    if d_model % 2 != 0:
        raise ValueError("Use an even d_model for this teaching implementation.")

    positions = torch.arange(max_length, device=device).unsqueeze(1)
    dimension_pairs = torch.arange(0, d_model, 2, device=device)
    frequencies = torch.exp(
        -math.log(10000.0) * dimension_pairs / d_model
    )

    encoding = torch.zeros(max_length, d_model, device=device)
    encoding[:, 0::2] = torch.sin(positions * frequencies)
    encoding[:, 1::2] = torch.cos(positions * frequencies)
    return encoding


positional = sinusoidal_positional_encoding(max_length=20, d_model=8)
positional[:4]

tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
          9.9995e-01,  1.0000e-03,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
          9.9980e-01,  2.0000e-03,  1.0000e+00],
        [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
          9.9955e-01,  3.0000e-03,  1.0000e+00]])

In [4]:
for dimension in range(4):
    plt.plot(positional[:, dimension].numpy(), label=f"dimension {dimension}")

plt.xlabel("Position")
plt.ylabel("Encoding value")
plt.title("Sinusoidal positional encoding")
plt.legend()
plt.show()

/tmp/ipykernel_735/1039428497.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
positioned_tokens = token_vectors + positional[: token_ids.shape[1]].unsqueeze(0)
print("Embeddings + positions:", tuple(positioned_tokens.shape))

Embeddings + positions: (1, 4, 8)


## 3. Residual connection and layer normalization

Layer normalization is calculated across a token's feature dimension.

In [6]:
x = torch.tensor([[1.0, 2.0, 3.0]])
sublayer_output = torch.tensor([[0.5, -0.5, 1.0]])
residual = x + sublayer_output

mean = residual.mean(dim=-1, keepdim=True)
variance = residual.var(dim=-1, keepdim=True, unbiased=False)
manual_normalized = (residual - mean) / torch.sqrt(variance + 1e-5)

layer_norm = nn.LayerNorm(3, elementwise_affine=False)
torch_normalized = layer_norm(residual)

print("Residual:", residual)
print("Manual LayerNorm:", manual_normalized)
print("PyTorch LayerNorm:", torch_normalized)
assert torch.allclose(manual_normalized, torch_normalized, atol=1e-5)

Residual: tensor([[1.5000, 1.5000, 4.0000]])
Manual LayerNorm: tensor([[-0.7071, -0.7071,  1.4142]])
PyTorch LayerNorm: tensor([[-0.7071, -0.7071,  1.4142]])


## 4. Position-wise feed-forward network

Attention mixes information across tokens. The FFN then transforms each token
independently using the same parameters at every position.

In [7]:
class PositionWiseFFN(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


ffn = PositionWiseFFN(d_model=8, d_ff=32)
ffn_output = ffn(positioned_tokens)

print("FFN input:", tuple(positioned_tokens.shape))
print("FFN output:", tuple(ffn_output.shape))

FFN input: (1, 4, 8)
FFN output: (1, 4, 8)


## 5. A complete post-norm encoder layer

The implementation follows:

`self-attention → residual + norm → FFN → residual + norm`

In [8]:
class TeachingEncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int):
        super().__init__()
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = PositionWiseFFN(d_model, d_ff)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(
        self,
        x: torch.Tensor,
        key_padding_mask: torch.Tensor | None = None,
    ):
        attention_output, attention_weights = self.attention(
            x,
            x,
            x,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            average_attn_weights=False,
        )
        x = self.norm1(x + attention_output)
        ffn_output = self.ffn(x)
        y = self.norm2(x + ffn_output)
        return y, attention_weights


encoder_layer = TeachingEncoderLayer(d_model=8, num_heads=2, d_ff=32)
encoder_output, attention_weights = encoder_layer(positioned_tokens)

print("Encoder output:", tuple(encoder_output.shape))
print("Attention weights:", tuple(attention_weights.shape))

Encoder output: (1, 4, 8)
Attention weights: (1, 2, 4, 4)


## 6. Stack encoder layers

Repeated encoder layers preserve the `(batch, sequence, d_model)` shape.

In [9]:
class TeachingEncoder(nn.Module):
    def __init__(self, num_layers: int, d_model: int, num_heads: int, d_ff: int):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                TeachingEncoderLayer(d_model, num_heads, d_ff)
                for _ in range(num_layers)
            ]
        )

    def forward(self, x: torch.Tensor):
        all_attention = []
        for layer in self.layers:
            x, weights = layer(x)
            all_attention.append(weights)
        return x, all_attention


encoder = TeachingEncoder(num_layers=3, d_model=8, num_heads=2, d_ff=32)
final_output, layer_attention = encoder(positioned_tokens)

print("Final output:", tuple(final_output.shape))
print("Number of attention tensors:", len(layer_attention))

Final output: (1, 4, 8)
Number of attention tensors: 3


## Practice

1. Compare the same token at two different positions.
2. Remove positional encodings and permute the input tokens.
3. Change `d_ff` from 32 to 8 and compare parameter counts.
4. Add `<PAD>` tokens and pass a Boolean `key_padding_mask`.
5. Replace post-norm with pre-norm and write down the new order.